In [ ]:
import torch
import numpy as np
from PIL import Image
import shutil
import os
os.chdir('../')
from cxas.segmentor import CXAS
import pandas as pd


In [ ]:
import cxas

In [ ]:
model = CXAS(
    model_name = 'UNet_ResNet50_default',
    gpus       = 'cpu'
)

In [ ]:
# path = '/Users/constantinseibold/Downloads/CXAS_CX_DATA_CURATED_2025-04-12/cxas_pe_1/Cbs2685/Thorax - 3KLEFEI2OQHZ/Thorax_pa_1/IM-0003-0001.dcm'

path = '/Users/constantinseibold/Downloads/CXAS_CX_DATA_CURATED_2025-04-12/cxas_pe_1/Cbs2685/Thorax - 3KLEFEI2OQHZ/Thorax_pa_1/'

# model.extract_features_for_file(path, 'radiomics', do_store=True)
model.extract_features_for_folder(path, './here','radiomics', do_store=True, create=True)

In [ ]:

sample_output = model.process_file(
        filename = path
        )
mask = sample_output['segmentation_preds']


In [ ]:
import numpy as np
import SimpleITK as sitk
import pandas as pd
from radiomics import featureextractor
import os
import torch.nn.functional as F
from tqdm import tqdm

image_sitk = sitk.ReadImage(path)
image_array = sitk.GetArrayFromImage(image_sitk)
image_array = F.interpolate(torch.tensor(image_array).unsqueeze(0).float(), (512,512)).numpy()[0]


image_sitk = sitk.GetImageFromArray(image_array)

mask_np = F.interpolate(mask.float(), image_array.shape[1:], mode='nearest').numpy()[0]

# Initialize PyRadiomics feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.enableAllFeatures()
extractor.enableAllImageTypes()


# Store all features
all_features = []

for class_idx in tqdm(range(mask_np.shape[0])):
    mask_channel = mask_np[class_idx:class_idx+1]

    # Skip empty masks
    if not mask_channel.sum():
        continue

    mask_sitk = sitk.GetImageFromArray(mask_channel.astype(np.uint8))

    # You can pass a label, but since it's binary, label=1 works
    result = extractor.execute(image_sitk, mask_sitk, label=1)

    # Add class info and flatten dict
    result_flat = {'class': class_idx}
    result_flat.update(result)

    all_features.append(result_flat)

# Convert to DataFrame
df = pd.DataFrame(all_features)

df

In [ ]:
list(df.keys())

In [ ]:
input_path = '/Users/constantinseibold/Downloads/CXAS_CX_DATA_CURATED_2025-04-12'
out_path = '/Users/constantinseibold/workspace/research/matthias_cxas/xray_results3'

model.extract_features_for_folder(
    input_directory_name = input_path,  
    output_directory = out_path,
    feat_to_extract = 'area',
    create = True, 
)

pd.read_csv('./out_feats/images.csv')

# Extract Features

A list of all extractable features is provided [here](ChestXRayAnatomySegmentation/docs/available_features).

In [ ]:
path = 'images/126_IM-0176-2002.dcm'

features = model.extract_features_for_file(
    filename = path,
    feat_to_extract = 'CTR',
    draw = True,
)
print(features['score'])
features['drawing']


In [ ]:
path = 'images/126_IM-0176-2002.dcm'

features = model.extract_features_for_file(
    filename = path,
    feat_to_extract = 'SCD',
    draw = True,
)
print('The SCD'features['score'])
features['drawing']


# Extract Features for folder